# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [5]:
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

print(df_fossil.groupby('Region')['TWh'].sum().sort_values(ascending=False).round(0))
print(df_fossil.groupby('Country')['TWh'].sum().nlargest(5).round(0))
print(df_fossil.groupby('Country')['Share_pct'].sum().nlargest(5))

Region
Asia             35045.0
Europe           15387.0
Latin America    12376.0
North America    11131.0
Middle East       9671.0
Africa            8620.0
Oceania           8142.0
Name: TWh, dtype: float64
Country
Saudi Arabia    9671.0
Japan           9442.0
China           9421.0
South Africa    8620.0
South Korea     8252.0
Name: TWh, dtype: float64
Country
Saudi Arabia    99
Mexico          88
South Africa    88
China           86
Japan           85
Name: Share_pct, dtype: int64


In [6]:
COAL, OIL, GAS = '#0072B2', '#E69F00', '#009E73'   # colour-blind-safe

fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map={'Coal': COAL, 'Oil': OIL, 'Natural Gas': GAS,
                        '(?)': '#D9D9D9'},   # '(?)' = parent nodes, drawn grey
)

fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='%{label}<br>%{value:,.0f} TWh<extra></extra>',
    textfont=dict(size=13),
)
fig.update_layout(
    title='YOUR INSIGHT TITLE HERE',
    margin=dict(t=70, l=10, r=10, b=10),
    width=1000, height=650,
)
fig.show()

## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [7]:
df_low = (df.loc[df['Source_Type'] == 'Low-carbon']
            .groupby('Country', as_index=False)['TWh'].sum()
            .sort_values('TWh', ascending=False))
df_low['All'] = 'Low-carbon'      # dummy root node

print(df_low.round(0).head(8))

          Country     TWh         All
4          France  7593.0  Low-carbon
9          Norway  6174.0  Low-carbon
2          Canada  5969.0  Low-carbon
1          Brazil  4975.0  Low-carbon
11    South Korea  2484.0  Low-carbon
6           India  1336.0  Low-carbon
13  United States  1137.0  Low-carbon
7           Japan  1012.0  Low-carbon


In [8]:
LOWC = '#0072B2'

fig_tm = px.treemap(df_low, path=['All', 'Country'], values='TWh',
                    color_discrete_sequence=[LOWC])
fig_tm.update_traces(texttemplate='%{label}<br>%{value:,.0f} TWh',
                     hovertemplate='%{label}<br>%{value:,.0f} TWh<extra></extra>')
fig_tm.update_layout(title='Low-carbon energy (Nuclear + Hydro) by country, TWh',
                     margin=dict(t=60, l=10, r=10, b=10), width=900, height=550)
fig_tm.show()

In [9]:
bar = df_low.sort_values('TWh')     # smallest first, so the largest ends up on top

fig_bar = px.bar(bar, x='TWh', y='Country', orientation='h',
                 text=bar['TWh'].round(0).map('{:,.0f} TWh'.format),
                 color_discrete_sequence=[LOWC])
fig_bar.update_traces(textposition='outside', cliponaxis=False)
fig_bar.update_layout(
    title='YOUR INSIGHT TITLE HERE',
    plot_bgcolor='white',
    xaxis=dict(title='Low-carbon energy (TWh)', range=[0, bar['TWh'].max() * 1.15],
               showgrid=False),
    yaxis=dict(title=None),
    width=900, height=max(450, 30 * len(bar)),
    font=dict(size=13))
fig_bar.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

*YOUR ANSWER HERE*

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [10]:
reg = df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh'].sum()
reg['share'] = reg['TWh'] / reg.groupby('Region')['TWh'].transform('sum') * 100

pivot = reg.pivot(index='Region', columns='Source', values='share').fillna(0)
print(pivot.sort_values('Coal', ascending=False).round(0))

Source         Coal  Natural Gas   Oil
Region                                
Africa         78.0          5.0  16.0
Asia           49.0         24.0  27.0
Oceania        36.0         32.0  32.0
North America  14.0         36.0  50.0
Latin America  10.0         33.0  57.0
Europe          6.0         51.0  44.0
Middle East     0.0         47.0  53.0


In [11]:
region_order = pivot['Coal'].sort_values().index.tolist()   # lowest first → highest at top

reg['label'] = np.where(reg['Source'] == 'Coal',
                        reg['share'].round(0).astype(int).astype(str) + '%', '')

fig3 = px.bar(
    reg, x='share', y='Region', color='Source', orientation='h', text='label',
    color_discrete_map={'Coal': COAL, 'Oil': '#BDBDBD', 'Natural Gas': '#8C8C8C'},
    category_orders={'Source': ['Coal', 'Oil', 'Natural Gas'],
                     'Region': region_order},
)
fig3.update_traces(textposition='inside', textfont=dict(color='white', size=13))
fig3.update_layout(
    title='YOUR INSIGHT TITLE HERE',
    plot_bgcolor='white',
    xaxis=dict(title='Share of region’s fossil energy (%)', range=[0, 100],
               showgrid=False),
    yaxis=dict(title=None),
    legend=dict(orientation='h', y=1.08, x=0, title=None),
    width=900, height=450, font=dict(size=13))
fig3.show()

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

*YOUR ANSWER HERE*